# Анализ лояльности пользователей Яндекс Афиши

In [ ]:
#Установка зависимостей
!pip install -r requirements.txt

# 1. Загрузка данных и их предобработка

## Задача 1.1  

Требуется написать SQL-запрос, выгружающий в датафрейм pandas необходимые данные из базы данных `data-analyst-afisha`:

- `user_id` — уникальный идентификатор пользователя, совершившего заказ;
- `device_type_canonical` — тип устройства, с которого был оформлен заказ (`mobile` — мобильные устройства, `desktop` — стационарные);
- `order_id` — уникальный идентификатор заказа;
- `order_dt` — дата создания заказа;
- `order_ts` — дата и время создания заказа;
- `currency_code` — валюта оплаты;
- `revenue` — выручка от заказа;
- `tickets_count` — количество купленных билетов;
- `days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- `event_id` — уникальный идентификатор мероприятия;
- `service_name` — название билетного оператора;
- `event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);
- `region_name` — название региона, в котором прошло мероприятие;
- `city_name` — название города, в котором прошло мероприятие.

In [ ]:
# Импорты
from dotenv import load_dotenv
import os  
import pandas as pd
from sqlalchemy import create_engine

In [ ]:
# Чтение .env файла
load_dotenv(dotenv_path='.env')

In [ ]:
# Подключение к бд
engine = create_engine(os.getenv('DB_CONNECTION_STRING')) 

In [ ]:
# Получение данных
query = '''
SELECT
  user_id,
  device_type_canonical,
  order_id,
  created_dt_msk AS order_dt,
  created_ts_msk AS order_ts,
  currency_code,
  revenue,
  tickets_count,
  created_dt_msk::date - LAG(created_dt_msk::date) OVER (
    PARTITION BY user_id
    ORDER BY created_dt_msk
  ) AS days_since_prev,
  event_id,
  service_name,
  event_type_main,
  r.region_name,
  c.city_name
FROM afisha.purchases
LEFT JOIN afisha.events e USING (event_id) 
LEFT JOIN afisha.city c USING (city_id)
LEFT JOIN afisha.regions r USING (region_id)
WHERE device_type_canonical IN ('mobile', 'desktop') 
  AND event_type_main != 'фильм'
ORDER BY user_id
'''
df = pd.read_sql_query(query, con=engine)

In [ ]:
df


## Задача 1.2 

Изучить общую информацию о выгруженных данных. Оценить корректность выгрузки и объём полученных данных.

Предположить, какие шаги необходимо сделать на стадии предобработки данных — например, скорректировать типы данных.

Зафиксировать основную информацию о данных в кратком промежуточном выводе.

In [ ]:
# Размерность
df.shape

In [ ]:
# Пропуски
df.isna().sum()

In [ ]:
# Проверка типов
df.info()

### Промежуточный вывод

Данные корректно получены из базы данных. Строк: 290611, колонок: 14

Пропуски найдены только в столбце `days_since_prev` - это ожидаемое поведение

Требуются следующие корректировки типов данных:
- Столбец days_since_prev привести к int
- Downcast числовых столбцов для уменьшения затрат памяти

# 2. Предобработка данных

## Задача 2.1

Данные о выручке сервиса представлены в российских рублях и казахстанских тенге. Требуется привести выручку к единой валюте — российскому рублю.

Для этого можно использовать датасет с информацией о курсе казахстанского тенге по отношению к российскому рублю за 2024 год — `final_tickets_tenge_df.csv`. Его можно загрузить по пути `https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')`

Значения в рублях представлено для 100 тенге.

Результаты преобразования сохранить в новый столбец `revenue_rub`.

In [ ]:
tenge_rate = pd.read_csv('https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv', parse_dates=['data'])
tenge_rate = tenge_rate.set_index('data')

In [ ]:
tenge_rate

In [ ]:
df['revenue_rub'] = df.apply(
    lambda row: 
        row['revenue'] 
        if row['currency_code'] == 'rub' 
        else row['revenue'] * tenge_rate.loc[row['order_dt']]['curs'] / tenge_rate.loc[row['order_dt']]['nominal'],
    axis=1
)

In [ ]:
df

## Задача 2.2

### 2.2.1 Проверить на пропущенные значения

In [ ]:
df.isna().sum()

Пропуски найдены только в столбце `days_since_prev` - это ожидаемое поведение

### 2.2.2 Преобразовать типы данных

In [ ]:
# Столбец days_since_prev привести к int
df['days_since_prev'] = pd.to_numeric(df['days_since_prev'])

# Downcast числовых столбцов для уменьшения затрат памяти
for col in df.columns:  
	if pd.api.types.is_integer_dtype(df[col]):  
		df[col] = pd.to_numeric(df[col], downcast='integer')  
	elif pd.api.types.is_float_dtype(df[col]):  
		df[col] = pd.to_numeric(df[col], downcast='float')

In [ ]:
df.info()

### 2.2.3 Изучить значения в ключевых столбцах

In [ ]:
df['service_name'].value_counts()

In [ ]:
df['event_type_main'].value_counts()

In [ ]:
df['region_name'].value_counts()

In [ ]:
df['city_name'].value_counts()